In [ ]:
'''
RFP - 생성 파이프라인

공공입찰 RFP 문서를 대상으로, 사용자 질문에 대해 관련 문서를 찾아 근거 기반으로 답을 생성하는 파이프라인.

핵심 흐름
1. 데이터 로드: 사전 청킹된 chunks.pkl, KURE-v1 임베딩 인덱스 로드
2. 문서 힌트 추출: 질문에서 언급된 발주기관/사업명을 파일명과 매칭해 관련 문서를 좁힘
3. 조건 필터링: 금액, 지자체 여부, 공사 등 구조화된 조건 추출 및 적용
4. 질문 유형별  컨텍스트 구성: 집계형/비교형/단일문서/필터검색 등으로 분기
5. 답변 생성: gpt-5-mini + 기권 규칙이 포함된 시스템 프롬프트

포함 내용
- 문서 힌트 추출, 필터링, 검색 로직
- 답변 생성 함수 ask_rfp_final
- 표/그림 기반 질문(visual QA) 및 코퍼스 통계형(analytics) 질문 테스트

'''

In [ ]:
# 환경 설정 및 라이브러리 설치

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install faiss-cpu sentence-transformers openai -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 55.8 MB/s eta 0:00:00


In [ ]:
# 청크 데이터 로드

import sys
import types
import pickle
import re
import numpy as np
from pathlib import Path


src_module = types.ModuleType('src')
chunking_module = types.ModuleType('src.chunking')

class Chunk:
    pass

chunking_module.Chunk = Chunk
src_module.chunking = chunking_module
sys.modules['src'] = src_module
sys.modules['src.chunking'] = chunking_module

DATA_DIR = Path('/content/drive/MyDrive/중급 프로젝트')

with open(DATA_DIR / 'chunks.pkl', 'rb') as f:
    chunk_objects = pickle.load(f)

all_chunks_final = []
chunk_metadata_final = []

for c in chunk_objects:
    all_chunks_final.append(c.text)
    meta_info = c.metadata if c.metadata else {}
    chunk_metadata_final.append({
        '파일명': c.doc_id,
        '발주기관': meta_info.get('발주_기관', ''),
        '사업금액': meta_info.get('사업_금액', None),
        '마감일': meta_info.get('입찰_참여_마감일', ''),
    })

print(f"청크 로드 완료: {len(all_chunks_final)}개")

청크 로드 완료: 18239개


In [ ]:
import faiss

with open(DATA_DIR / 'kure_embeddings.pkl', 'rb') as f:
    kure_embeddings = pickle.load(f)

index_kure = faiss.IndexFlatL2(kure_embeddings.shape[1])
index_kure.add(np.array(kure_embeddings).astype('float32'))

print(f"KURE 인덱스: {index_kure.ntotal}개")

KURE 인덱스: 18239개


In [ ]:
# 임베딩 모델 로드

import torch
from sentence_transformers import SentenceTransformer

kure_model = SentenceTransformer('nlpai-lab/KURE-v1', device='cuda', model_kwargs={'torch_dtype': torch.float16})
print(kure_model.device)

In [ ]:
# Generation 파이프라인 패키지 로드

with open(DATA_DIR / 'generation_final_pipeline.pkl', 'rb') as f:
    final_package = pickle.load(f)

print(final_package['summary'])
print()
print(list(final_package['function_sources'].keys()))

{'embedding_model': 'nlpai-lab/KURE-v1', 'data_source': 'chunks.pkl (merged_docs.pkl 기반, 18239개 청크)', 'prompt_version': 'SYSTEM_PROMPT_NEW_V2 (기권 규칙 포함 최종본)', 'test_result': {'golden_set_40 (dev_refined_review-candidate)': '40/40', 'single_doc': '10/10', 'multi_doc_compare': '10/10', 'follow_up': '10/10', 'unknown_abstain': '10/10', 'classic_issues (재난혼동/OO공사/집계형)': '3/3'}}

['ask_rfp_final', 'extract_doc_hints_multi', 'find_relevant_keywords', 'is_aggregation_question', 'extract_filter_conditions', 'get_filtered_candidates', 'search_with_filter', 'normalize_org_name']


In [ ]:
# OpenAI 클라이언트 설정

from google.colab import userdata
import openai

api_key = userdata.get('OPENAI_API_KEY')
client = openai.OpenAI(api_key=api_key)

In [ ]:
seen = set()
all_filenames_with_biz = []
for cm in chunk_metadata_final:
    if cm['파일명'] not in seen:
        seen.add(cm['파일명'])
        all_filenames_with_biz.append((cm['파일명'], cm.get('발주기관', '')))

print(f"고유 문서 수: {len(all_filenames_with_biz)}")

고유 문서 수: 98


In [ ]:
# 기관명 별칭 매핑 / 공통 단어 블랙리스트 / 법률 키워드 맵

ORG_ALIAS_MAP = {
    '대검찰청': ['검찰'],
    '고려대학교': ['고려대'],
    '한국산업단지공단': ['산단'],
}

COMMON_SUFFIX_WORDS = {
    '박물관', '시스템', '센터', '공단', '진흥원', '협회', '재단', '연구원', '공사', '대학교',
    '사업', '관리', '운영', '구축', '개선', '개발', '지원', '정보', '용역', '기관', '기술',
    '고도화', '확대', '기능', '서비스', '일자리', '플랫폼', '통합', '접수',
    '일자리재단', '일자리플랫폼', '보험', '입찰공고', '공고',
    '과학연구', '과학연', '학연구', '연구소', '기록관리', '경기기록',
    '학교', '학교 ', ' 학교', '산학협력단', '산학협력', '학협력단',
    '통합시스템'
}
COMMON_FILENAME_WORDS = COMMON_SUFFIX_WORDS | {'용역', '수립', '2차', '1차', '3차', '운영', '및', '구축용역'}

LEGAL_KEYWORDS_MAP = {
    '하도급': ['하도급'],
    '공동수급': ['공동수급', '지분율', '컨소시엄'],
    '지분율': ['지분율', '공동수급'],
    '계약보증금': ['계약보증금', '보증금'],
    '평가': ['배점', '평가비율', '기술평가', '가격평가'],
    '제안서 보상': ['제안서 보상'],
    '불이익': ['부정당업자', '입찰보증금', '귀속'],
    '제출물': ['제출서류', '부', 'USB', '제출규격'],
    '제출': ['제출서류', 'USB'],
    '수량': ['부', 'USB'],
    '구축기간': ['사업기간', '구축기간', '개월'],
    '사업기간': ['사업기간', '구축기간', '개월'],
    '유지보수': ['무상유지보수', '유지보수기간', '하자보수', '무상 하자보수'],
    '참가자격': ['참가자격', '참가 자격'],
    '유지관리': ['하자보수', '유지관리 인력', '무상 하자보수'],
    '교육 의무': ['유지관리 인력', '사용자 및 관리자', '하자보수'],
    '교육을': ['유지관리 인력', '사용자 및 관리자', '하자보수'],
    '검수 후': ['하자보수', '유지관리 인력'],
    '재입찰': ['재입찰', '재공고입찰', '최초의 입찰'],
    '재공고': ['재입찰', '재공고입찰', '최초의 입찰'],
    '조건 변경': ['재입찰', '재공고입찰', '최초의 입찰'],
    '지역 요건': ['주된 영업소', '소재지'],
    '부산에': ['주된 영업소', '소재지'],
    '지역요건': ['주된 영업소', '소재지'],
    '소재지': ['주된 영업소', '소재지'],
    '보유인력': ['보유인력', '배점한도'],
    '배점한도': ['보유인력', '배점한도'],
    '계량평가': ['보유인력', '배점한도', '재무구조'],
    '규모비율': ['규모비율', '환산점수', '점수비중'],
    '환산점수': ['규모비율', '환산점수', '점수비중'],
    '수행실적': ['규모비율', '환산점수', '수행실적'],
    '신인도': ['신인도', '가점'],
    '가점표': ['신인도', '가점'],
    '연구원 승인': ['Lesson', '회람'],
    '발생한 경우': ['Lesson', '회람'],
    '회람': ['Lesson', '회람'],
}

In [ ]:
# 청킹 파편화로 원본 표가 깨진 문서를 수동으로 보정한 텍스트

items_list = [
    "1. 장애인 기업(중소벤처기업부 발급)", "2. 여성기업(중소벤처기업부 발급)", "3. 중증장애인생산품 생산시설(보건복지부 지정)",
    "4. 사회적 기업(고용노동부 지정)", "5. 예비 사회적 기업(지방자치단체 지정)", "6. 사회적협동조합(정부부처 지정)",
    "7. 자활기업(지방자치단체 지정)", "8. 가족친화 우수기업", "9. 하도급거래 모범기업",
    "10. 노사문화 우수기업", "11. 남녀고용평등 우수기업", "12. 모범납세자"
]
scores_list = [1, 1, 1, 1, 1, 1, 1, 0.8, 0.8, 0.5, 0.5, 0.3]

matched_table = "\n".join(f"{item} : {score}점" for item, score in zip(items_list, scores_list))
print(matched_table)

CORRECTED_TABLES = {
    ('서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf', '약자기업'):
        "[신인도 가점표 - 약자기업 지원 및 정책적 지원 항목별 점수 (정확히 매칭됨)]\n" + matched_table
}
print("\n보정 데이터 저장 완료")

In [ ]:
# 질문 → 관련 키워드 확장

def find_relevant_keywords(question):
    matched = []
    for trigger, kws in LEGAL_KEYWORDS_MAP.items():
        if trigger in question:
            matched.extend(kws)
    return list(set(matched))

# 집계형 질문 판별

def is_aggregation_question(question):
    keywords = ['몇 개', '개수', '다 나열', '몇 건']
    strong_total = '전부' in question or ('총' in question and ('개' in question or '건' in question))
    return any(kw in question for kw in keywords) or strong_total

def extract_filter_conditions(query):
    conditions = {}
    if '억' in query and ('이상' in query or '넘는' in query):
        match = re.search(r'(\d+)억', query)
        if match:
            conditions['금액_최소'] = int(match.group(1)) * 100000000
    if '지자체' in query or '지방자치단체' in query:
        conditions['지자체'] = True
    if '공사' in query and ('OO공사' in query or '발주기관이' in query):
        conditions['공사'] = True
    if 'AI' in query:
        conditions['주제_AI'] = True
    if '긴급' in query:
        conditions['긴급'] = True
    if '보안' in query:
        conditions['보안'] = True
    if '재난' in query:
        conditions['재난'] = True
    return conditions

# 발주기관명이 정확히 행정구역명(시/도/군/구 등)으로 끝나는 경우만 지자체로 판별

def is_local_gov(org):
    if org is None or (isinstance(org, float)):
        return False
    return bool(re.search(r'(광역시|특별시|특별자치도|특별자치시|[가-힣]+도|[가-힣]+시|[가-힣]+군|[가-힣]+구)$', str(org).strip()))

def get_filtered_candidates(conditions, chunk_metadata):
    if not conditions:
        return None
    doc_info = {}
    for cm in chunk_metadata:
        fname = cm['파일명']
        if fname not in doc_info:
            doc_info[fname] = cm
    allowed = set()
    for fname, info in doc_info.items():
        ok = True
        if '금액_최소' in conditions:
            amt = info.get('사업금액')
            if amt is None or amt < conditions['금액_최소']:
                ok = False
        if conditions.get('지자체'):
            if not is_local_gov(info.get('발주기관')):
                ok = False
        if conditions.get('공사'):
            org = str(info.get('발주기관', ''))
            if '공사' not in org:
                ok = False
        if conditions.get('긴급'):
            if '긴급' not in fname:
                ok = False
        if conditions.get('재난'):
            if '재난' not in fname:
                ok = False
        if ok:
            allowed.add(fname)
    return allowed if allowed else None

def search_with_filter(query, index, model, chunk_metadata, all_chunks, k=10, max_per_doc=1):
    conditions = extract_filter_conditions(query)
    allowed_filenames = get_filtered_candidates(conditions, chunk_metadata)
    query_embedding = model.encode([query])
    search_k = min(len(all_chunks), 2000)
    distances, indices = index.search(np.array(query_embedding).astype('float32'), search_k)
    seen_docs = {}
    results = []
    for i in indices[0]:
        doc_name = chunk_metadata[i]['파일명']
        if allowed_filenames is not None and doc_name not in allowed_filenames:
            continue
        count = seen_docs.get(doc_name, 0)
        if count < max_per_doc:
            results.append(i)
            seen_docs[doc_name] = count + 1
        if len(results) >= k:
            break
    return results

def normalize_org_name(name):
    return re.sub(r'(특별시|광역시|특별자치시|특별자치도)', '', name)

In [ ]:
# 문서 힌트 추출
# 질문에서 어떤 특정 문서를 가리키는지 여러 단계로 추론
#  1단계: 발주기관명이 질문에 언급됐는지 매칭
#  2단계: 따옴표로 감싼 사업명 / 영문 키워드가 사업명과 일치하는지 확인
#  3단계: 위 두 단계로 못 찾으면 파일명 키워드 fuzzy 매칭으로 후보 스코어링
#  4단계: 같은 발주기관에 문서가 여러 개 매칭되면, 질문 키워드와 가장 많이 겹치는 문서 하나만 최종 선택

def extract_doc_hints_multi(question, all_filenames_with_biz):
    q_no_space = question.replace(' ', '').replace('&', '')
    org_candidates = []
    for fname, biz_name in all_filenames_with_biz:
        org_part = fname.replace('refined_', '').split('_')[0].strip()
        org_core = re.sub(r'\s*\(.*?\)\s*', '', org_part).strip()
        org_core_clean = re.sub(r'^\(사\)', '', org_core).strip()
        org_core_clean = re.sub(r'\s*입찰공고\s*$', '', org_core_clean).strip()
        org_core_norm = normalize_org_name(org_core_clean)
        if len(org_core_clean) < 2:
            continue
        matched = False
        if org_core_clean in question:
            matched = True
        elif len(org_core_norm) >= 3 and org_core_norm in question:
            matched = True
        elif org_core_clean in ORG_ALIAS_MAP and any(alias in question for alias in ORG_ALIAS_MAP[org_core_clean]):
            matched = True
        else:
            min_len = 4
            for target_str in [org_core_clean, org_core_norm]:
                for start in range(len(target_str) - min_len + 1):
                    for length in range(len(target_str) - start, min_len - 1, -1):
                        substr = target_str[start:start+length]
                        if substr.strip() in question and substr.strip() not in COMMON_SUFFIX_WORDS:
                            matched = True
                            break
                    if matched:
                        break
                if matched:
                    break
        if matched:
            org_candidates.append((fname, org_core_clean))
    
    biz_candidates = []
    quoted = re.findall(r"['\"]([^'\"]+)['\"]", question)
    for fname, biz_name in all_filenames_with_biz:
        biz_name = str(biz_name).strip()
        if len(biz_name) >= 4 and biz_name in question:
            biz_candidates.append(fname)
            continue
        for q in quoted:
            if q in biz_name or biz_name in q:
                biz_candidates.append(fname)
                break
        eng_words = re.findall(r'[A-Za-z][A-Za-z&\s]{2,}[A-Za-z]', biz_name)
        for ew in eng_words:
            ew_no_space = ew.strip().replace(' ', '').replace('&', '')
            if len(ew_no_space) >= 4 and ew_no_space in q_no_space:
                biz_candidates.append(fname)
                break
    
    stopwords_general = {'사업의', '사업에서', '사업은', '어떻게', '되나요', '되나요?', '몇', '어떤', '얼마', '비교', '알려줘', '정리해줘', '무엇인가요', '관련', '입찰공고일', '공고일', '입찰공고'}
    raw_keywords = [w.rstrip('.,?!') for w in re.split(r'[ ,·]', question) if len(w) >= 4]
    keywords_all = [w for w in raw_keywords if w not in stopwords_general and w not in COMMON_FILENAME_WORDS and '입찰공고' not in w]
    
    def fuzzy_match(kw, text, min_overlap=4):
        kw_ns = kw.replace(' ', '')
        text_ns = text.replace(' ', '')
        if kw_ns in text_ns:
            return True
        for n in range(len(kw_ns), min_overlap - 1, -1):
            if kw_ns[:n] in text_ns:
                return True
        return False
    
    def keyword_weight(kw):
        return 3 if re.search(r'[A-Za-z]', kw) else 1
    
    filename_candidates = []
    for fname, biz_name in all_filenames_with_biz:
        fname_clean = fname.replace('refined_', '').replace('.hwp', '').replace('.pdf', '')
        matched_kws = [kw for kw in keywords_all if fuzzy_match(kw, fname_clean)]
        score = sum(keyword_weight(kw) for kw in matched_kws)
        if score > 0:
            filename_candidates.append((fname, score, len(matched_kws)))
    
    if filename_candidates:
        filename_candidates.sort(key=lambda x: -x[1])
        max_score = filename_candidates[0][1]
        for top_fname, score, cnt in filename_candidates:
            if score >= max_score * 0.6 or score >= 1:
                if top_fname not in [f for f, _ in org_candidates] and top_fname not in biz_candidates:
                    if len(filename_candidates) <= 3 or score >= max(max_score * 0.6, 1):
                        biz_candidates.append(top_fname)
    
    org_groups = {}
    for fname, org_core in org_candidates:
        org_groups.setdefault(org_core, []).append(fname)
    
    stopwords = {'사업의', '사업에서', '어떻게', '되나요?', '되나요', '몇', '어떤', '얼마', '비교'}
    keywords = [w for w in re.split(r'[ ,]', question) if len(w) >= 2 and w not in stopwords]
    
    final_hints = []
    for org_core, fnames in org_groups.items():
        fnames = list(set(fnames))
        if len(fnames) == 1:
            final_hints.append(fnames[0])
        else:
            fname_to_biz = dict(all_filenames_with_biz)
            best_doc, best_score2 = None, -1
            for fname in fnames:
                biz_name = fname_to_biz.get(fname, '')
                score2 = sum(1 for kw in keywords if kw in fname or kw in str(biz_name))
                if score2 > best_score2:
                    best_score2, best_doc = score2, fname
            final_hints.append(best_doc)
    
    for fname in biz_candidates:
        if fname not in final_hints:
            final_hints.append(fname)
    
    return list(dict.fromkeys(final_hints))

In [ ]:
# 프롬프트 규칙

SYSTEM_PROMPT_NEW_V2 = """
너는 'RFP 챗봇'이야. 입찰메이트 컨설턴트가 제안요청서(RFP) 문서를 빠르게 파악할 수 있게 도와줘.

## 기본 원칙

1. 반드시 아래에 제공된 문서 내용(컨텍스트)에 근거해서만 답변해. 문서에 없는 내용을 추측하거나 지어내지 마.

2. 답변은 간결하고 명확하게 작성해. 불필요한 서론 없이 핵심부터 답해.

3. 질문 유형에 따라 답변 형식을 다르게 해:
   - 단일 사실 조회 (예: "예산이 얼마야?") → 핵심 수치/사실 위주로 간결하게
   - 두 개 이상 비교 (예: "A랑 B 중 뭐가 더 커?") → 각 항목을 나란히 제시하고 비교 결론 제시
   - 목적/배경을 묻는 질문 → 관련 섹션을 요약해서 설명
   - 조건에 맞는 여러 문서를 찾는 질문 → 목록 형태로 정리

4. 이전 대화에서 언급된 문서나 주제가 있으면, 후속 질문("그럼 마감일은?" 등)은 같은 문서/주제 맥락에서 답변해.

5. 답변 끝에는 근거가 된 문서명을 명시해.

## 답변을 거절/기권해야 하는 경우 (매우 중요)

아래 경우에는 문서 안에서 관련 정보를 억지로 찾아서 답하려 하지 말고, 명확히 "답변할 수 없다"고만 말하고 끝내. 관련 있어 보이는 부가 정보를 나열하지 마.

- **범위 밖 요청(out_of_scope)**: 네가 할 수 없는 행동을 요청하는 경우(전화 걸기, 이메일 보내기, 실시간 조회 등), 또는 "오늘", "지금", "최신"처럼 실시간·최신 정보를 요구하는 경우. 이때는 "이 기능은 제가 수행할 수 없습니다" 또는 "실시간 정보는 제공된 문서에서 확인할 수 없습니다"라고만 답하고, 대신 관련 문서를 찾아주거나 연락처를 나열하는 등 다른 시도를 하지 마.

- **근거 부족(insufficient_evidence)**: 낙찰 결과, 경쟁사 현황, 예상 낙찰가처럼 애초에 이 문서(제안요청서)에 있을 수 없는 정보를 물어보는 경우. "확인되지 않습니다"라고만 답해.

- **판단/추측 요청(ambiguous)**: "우리 회사가 자격을 충족하는지 판정해줘", "수주 확률이 얼마냐" 처럼 사용자의 상황과 문서를 대조해서 네가 주관적으로 판단·확률을 계산해야 하는 질문. 이런 판정이나 확률 계산은 네가 할 수 없다고 답하고, 판단에 필요한 조건 목록만 간단히 안내해도 되지만 장황하게 체크리스트를 만들지는 마.

- **사용자가 임의의 가정을 세우고 그 가정으로 확정 답변을 요구하는 경우**: "문서에 없으면 OO라고 가정하고 확정해줘"처럼, 사용자가 제시한 임의의 규칙(추측)을 근거 삼아 사실인 것처럼 답을 만들어달라는 요청. 이건 절대 받아들이지 마. "문서에 없는 정보는 임의로 가정해서 확정할 수 없습니다"라고 답하고, 사용자가 제안한 가정을 그대로 적용해서 계산해주지 마.

## 표 형식 데이터 안내

컨텍스트에 [표]라는 표시와 함께 "항목 | 값" 형태로 된 부분이 나오면, 이는 원본 문서의 표를 옮긴 것이야. 각 줄은 표의 한 행을 의미하고, |로 구분된 각 항목은 표의 열(칸)을 의미해. 이 형식을 참고해서 항목과 값을 정확히 짝지어 답변해.

## 여러 문서 처리

컨텍스트에 여러 문서의 내용이 섞여 있을 수 있어. 각 문서 조각이 어느 문서(파일명)에서 왔는지 구분해서, 서로 다른 문서의 정보를 혼동하거나 섞어서 답하지 마.

일부 정보(예: 긴급 여부, 재공고 여부)는 본문 내용이 아니라 문서명(파일명)에만 표시되어 있을 수 있어. 문서명에 이런 정보가 있으면 그것도 근거로 활용해서 답해.

## 주제/카테고리 판단 시 주의사항

질문의 키워드와 문서 안의 유사한 단어가 겉보기에 비슷해 보여도, 실제 의미는 다를 수 있어. 문서의 실제 사업 목적과 내용까지 확인해서 질문 의도와 정확히 일치하는지 판단하고, 확신이 안 서면 "이 문서는 [실제 의미]를 다루고 있어 질문 의도와 다를 수 있습니다"처럼 구분해서 답해. 단어의 표면적 유사성만으로 포함시키지 마.

아래는 실제로 혼동이 발생했던 사례야. 반드시 참고해서 판단해:

예시: "재난 관련 사업을 찾아줘"라는 질문에, 사업명이 "적십자병원 병원정보 재해복구시스템 구축 용역"인 문서가 검색됐다고 하자. 이 문서는 재난(자연재해, 재난관리) 관련 사업이 아니야. "재해복구시스템(Disaster Recovery System)"은 서버/데이터베이스 장애 시 데이터를 복구하는 IT 인프라 용어이고, "병원정보시스템 데이터베이스 운영"을 다루는 순수 IT 시스템 구축 사업이야. 이 사업을 "재난 관련"으로 포함시키면 틀린 답변이야. 반드시 제외해.

마찬가지로 "응급의료 상황관리시스템"(병원 전원·환자 이송을 지원하는 IT 시스템)도 "재난 관리 시스템"과는 다른 목적의 사업이야. 재난은 지진, 홍수, 화재 등 자연재해나 사회재난에 대응하는 시스템을 뜻하며, 단순히 "응급", "긴급", "재해" 같은 단어가 사업명에 있다고 해서 재난 관련 사업으로 분류하면 안 돼.

## 질문 해석 관련

질문에 "OO", "XX" 같은 placeholder처럼 보이는 표현이 있어도, 이는 실제로 채워야 할 빈칸이 아니라 "특정 패턴을 가진 이름 전체"를 가리키는 일반적인 화법일 수 있어. 예를 들어 "발주기관이 OO공사인 사업"은 "발주기관명이 '공사'로 끝나는 모든 사업"을 뜻하는 것이지, 사용자가 실제 공사명을 지정해줘야 한다는 뜻이 아니야. 이런 경우 되묻지 말고, 컨텍스트 안에서 해당 패턴에 맞는 사업을 최대한 찾아서 답해.

## 금액 표기 관련

금액은 부가세(VAT) 포함/별도 표기가 문서마다 다를 수 있어. 답변할 때 원문에 표기된 형태(포함/별도 여부 포함) 그대로 전달하고, 임의로 환산하지 마.

## 구조화된 필드(공고번호, 사업금액, 입찰 참여 시작일/마감일, 발주기관) 답변 규칙

이 필드들은 컨설턴트의 실제 입찰 결정에 직결되니까 특히 신중하게 답해.

- 검색된 문서 조각과 메타데이터에 명확한 값이 있으면, 근거와 함께 답변해.
- 값이 없거나 불확실하면 절대 추정하지 말고 "확인되지 않습니다"라고 명확히 답해.
- 아래 함정에 특히 주의해:
  - 공고번호를 유사한 다른 번호나 제목의 "[재공고]" 표시만으로 추정하지 마.
  - 개찰 시각이나 제안서 평가 시각을 입찰 참여 마감일로 착각해서 답하지 마. 이 셋은 서로 다른 시점이야.
  - 공개일(공고가 게시된 날짜)을 입찰 참여 시작일로 대체하지 마.
  - 발주기관은 게시기관·수요기관·계약기관이 다를 수 있으니까, 근거 없이 하나를 임의로 선택하지 마.
  - 사업금액이 0원이나 1원으로 보이면, 이건 실제 금액이 아니라 비공개·미확정을 나타내는 표시일 수 있어. 이 경우 실금액처럼 답하지 말고 "금액이 비공개이거나 미확정 상태로 보입니다"라고 답해.

## 참가자격 / 제한조건 / 평가기준 / 제출요건 / 계약 리스크(위약금, 계약보증금 등) 답변 규칙

이 항목들도 컨설턴트가 실제로 입찰 여부를 판단하고 계약 의무를 이해하는 데 직결되니까 신중하게 답해.

- 검색된 문서 조각 안에 명확한 근거가 있을 때만 답변해.
- 명확한 근거가 없으면 "제공된 문서 범위에서는 확인되지 않습니다. 원문 전체 확인이 필요할 수 있습니다"라고 답해.
- 다른 사업의 일반적인 조항이나 통상적인 관행을 이 사업에 적용해서 답하지 마.

## 부분 정보 처리

질문에 여러 정보가 섞여 있고 그중 일부만 확인 가능하면, 확인되는 정보는 근거와 함께 답하고 확인 안 되는 정보만 위 규칙에 따라 "확인되지 않습니다"라고 답해. 일부가 확인 안 된다고 전체 답변을 포기하지 마.

## 컨텍스트 (검색된 문서 조각)
{context}

## 질문
{question}
"""

CORRECTED_TABLES = {
    ('서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf', '약자기업'):
        "[신인도 가점표 - 약자기업 지원 및 정책적 지원 항목별 점수 (정확히 매칭됨)]\n"
        "1. 장애인 기업(중소벤처기업부 발급) : 1점\n2. 여성기업(중소벤처기업부 발급) : 1점\n"
        "3. 중증장애인생산품 생산시설(보건복지부 지정) : 1점\n4. 사회적 기업(고용노동부 지정) : 1점\n"
        "5. 예비 사회적 기업(지방자치단체 지정) : 1점\n6. 사회적협동조합(정부부처 지정) : 1점\n"
        "7. 자활기업(지방자치단체 지정) : 1점\n8. 가족친화 우수기업 : 0.8점\n9. 하도급거래 모범기업 : 0.8점\n"
        "10. 노사문화 우수기업 : 0.5점\n11. 남녀고용평등 우수기업 : 0.5점\n12. 모범납세자 : 0.3점"
}

In [ ]:
# 최종 답변 생성 함수
# 질문 유형에 따라 컨텍스트 구성 방식 :
# CORRECTED_TABLES에 해당하는 표 보정 텍스트가 있으면 최우선으로 컨텍스트에 추가
#   1) 집계형 질문 + 문서 힌트 1개 → 그 문서의 청크 전부 사용
#   2) 문서 힌트 2개 이상 → 비교 질문으로 간주, 각 문서에서 관련 청크만 추림
#   3) 문서 힌트 1개 + 법률 키워드 있음 → 해당 문서 내 키워드 매칭 청크 우선
#   4) 문서 힌트 1개만 있음 → 그 문서 청크 상위 15개
#   5) 조건 필터만 있음 (특정 문서 아님) → 필터 통과 문서군에서 벡터 검색
#   6) 아무 힌트도 없음 → 전체 대상 기본 벡터 검색

def ask_rfp_final(question, model_name="gpt-5-mini", max_retries=2):
    doc_hints = extract_doc_hints_multi(question, all_filenames_with_biz)
    doc_hints = doc_hints[:3]
    keywords = find_relevant_keywords(question)
    conditions = extract_filter_conditions(question)
    allowed = get_filtered_candidates(conditions, chunk_metadata_final)
    
    fname_to_meta = {}
    for cm in chunk_metadata_final:
        if cm['파일명'] not in fname_to_meta:
            fname_to_meta[cm['파일명']] = cm
    
    def meta_header(fname):
        m = fname_to_meta.get(fname, {})
        org = m.get('발주기관', '')
        amt = m.get('사업금액')
        amt_str = f"{amt:,.0f}원" if amt not in (None, '') else "확인되지 않음"
        return f"[문서: {fname}]\n[발주기관(메타데이터): {org}]\n[사업금액(메타데이터): {amt_str}]"
    
    context_parts = []

    for (fname_key, kw_key), corrected_text in CORRECTED_TABLES.items():
        if fname_key in doc_hints and kw_key in question:
            context_parts.append(f"[문서: {fname_key}]\n{corrected_text}")
    
    if is_aggregation_question(question) and len(doc_hints) >= 1:
        stopwords_q = {'사업의', '사업에서', '어떻게', '되나요?', '되나요', '몇', '어떤', '얼마', '비교'}
        qkeywords = [w for w in re.split(r'[ ,]', question) if len(w) >= 2 and w not in stopwords_q]
        best_doc, best_score = doc_hints[0], -1
        for fname in doc_hints:
            biz = fname_to_meta.get(fname, {}).get('발주기관', '')
            score = sum(1 for kw in qkeywords if kw in fname or kw in str(biz))
            if score > best_score:
                best_score, best_doc = score, fname
        doc_hint = best_doc
        doc_chunks = [i for i, cm in enumerate(chunk_metadata_final) if cm['파일명'] == doc_hint]
        header = meta_header(doc_hint)
        for i in doc_chunks:
            context_parts.append(f"{header}\n{all_chunks_final[i]}")
    
    elif len(doc_hints) == 1 and keywords:
        doc_hint = doc_hints[0]
        doc_chunks = [i for i, cm in enumerate(chunk_metadata_final) if cm['파일명'] == doc_hint]
        keyword_chunks = [i for i in doc_chunks if any(kw in all_chunks_final[i] for kw in keywords)]
        result_indices = keyword_chunks[:25] if keyword_chunks else search_with_filter(question, index_kure, kure_model, chunk_metadata_final, all_chunks_final, k=10, max_per_doc=5)
        for i in result_indices:
            fname = chunk_metadata_final[i]['파일명']
            context_parts.append(f"{meta_header(fname)}\n{all_chunks_final[i]}")
    
    elif len(doc_hints) >= 2:
        for doc_hint in doc_hints:
            doc_chunks = [i for i, cm in enumerate(chunk_metadata_final) if cm['파일명'] == doc_hint]
            if keywords:
                matched = [i for i in doc_chunks if any(kw in all_chunks_final[i] for kw in keywords)]
                selected = matched[:8] if matched else doc_chunks[:8]
            else:
                selected = doc_chunks[:8]
            header = meta_header(doc_hint)
            for i in selected:
                context_parts.append(f"{header}\n{all_chunks_final[i]}")
    elif doc_hints:
        doc_hint = doc_hints[0]
        doc_chunks = [i for i, cm in enumerate(chunk_metadata_final) if cm['파일명'] == doc_hint]
        header = meta_header(doc_hint)
        for i in doc_chunks[:15]:
            context_parts.append(f"{header}\n{all_chunks_final[i]}")
    elif allowed:
        k = min(len(allowed), 80)
        result_indices = search_with_filter(question, index_kure, kure_model, chunk_metadata_final, all_chunks_final, k=k, max_per_doc=1)
        for i in result_indices:
            fname = chunk_metadata_final[i]['파일명']
            context_parts.append(f"{meta_header(fname)}\n{all_chunks_final[i]}")
    else:
        result_indices = search_with_filter(question, index_kure, kure_model, chunk_metadata_final, all_chunks_final, k=10, max_per_doc=3)
        for i in result_indices:
            fname = chunk_metadata_final[i]['파일명']
            context_parts.append(f"{meta_header(fname)}\n{all_chunks_final[i]}")
    
    context = "\n\n---\n\n".join(context_parts)
    final_prompt = SYSTEM_PROMPT_NEW_V2.format(context=context, question=question)
    
    for attempt in range(max_retries):
        response = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": final_prompt}],
            max_completion_tokens=8000,
            reasoning_effort="low"
        )
        answer = response.choices[0].message.content
        if answer:
            return answer
    return "(답변 생성 실패)"

In [ ]:
answer = ask_rfp_final("재난 관련 사업 중 예산이 5억 이상인 것만 알려줘")
print(answer)

- 봉화군 재난통합관리시스템 고도화 사업 — 사업금액: 900,000,000원(부가세 포함)

근거: 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급).hwp


In [ ]:
# 테스트 — 표/그림 등 시각자료 기반 질문

visual_test1 = [
    ("visual-hwp-table-001", "수문자료정보관리시스템(HDIMS) 재구축 용역(3단계)의 기술성 평가 구성표에서 정량적 평가와 정성적 평가의 소계 및 평가 방식은 각각 무엇인가?"),
    ("visual-hwp-table-002", "실시간통합연구비관리시스템(RCMS) 연계 모듈 변경 사업에서 입찰공고일 현재 4대 보험 납입 기준 보유인력이 정확히 20명인 업체는 보유인력 항목에서 몇 점을 받으며, 이 항목의 배점한도는 몇 점인가?"),
    ("visual-hwp-table-003", "국방과학연구소 기록관리시스템 사업의 요구사항 목록 전체를 기준으로, 요구사항 수가 가장 많은 구분과 가장 적은 구분(동률 포함)은 무엇이며 각각 몇 건인가? 총 요구사항 수도 함께 답하라."),
    ("visual-hwp-figure-001", "국방과학연구소 기록관리시스템의 대상 시스템 현황 그림에서 웹서버 제품·버전, DBMS 제품·버전, Active/Standby 운영서버의 운영체제를 각각 답하라."),
    ("visual-hwp-figure-002", "네팔 수자원관리 Pilot 시스템의 운영체계 그림에서 GIDC의 Main server와 DWRI의 Secondary server 사이 백업 주기와 DWRI 서버 유지보수 주기는 각각 어떻게 표시되어 있는가?"),
]

for cid, q in visual_test1:
    answer = ask_rfp_final(q)
    print(f"[{cid}] {q}")
    print(answer)
    print("\n")

[visual-hwp-table-001] 수문자료정보관리시스템(HDIMS) 재구축 용역(3단계)의 기술성 평가 구성표에서 정량적 평가와 정성적 평가의 소계 및 평가 방식은 각각 무엇인가?
정량적 평가
- 소계: 10점
- 평가방식: 절대평가
- 주요 항목(배점): 유사분야 수행실적 5점, 재무구조 및 경영상태(기업신용평가) 5점
근거: 한국수자원조사기술원_수문자료정보관리시스템(HDIMS) 재구축 용역(3단계).hwp

정성적 평가
- 소계: 90점
- 평가방식: 상대평가
- 세부항목(배점): 전략 및 방법론 20점, 기술 및 기능 20점, 성능 및 품질 20점, 프로젝트 관리 20점, 프로젝트 지원 10점
근거: 한국수자원조사기술원_수문자료정보관리시스템(HDIMS) 재구축 용역(3단계).hwp


[visual-hwp-table-002] 실시간통합연구비관리시스템(RCMS) 연계 모듈 변경 사업에서 입찰공고일 현재 4대 보험 납입 기준 보유인력이 정확히 20명인 업체는 보유인력 항목에서 몇 점을 받으며, 이 항목의 배점한도는 몇 점인가?
확인되지 않습니다.

제공된 문서들(광주과학기술원_실시간통합연구비관리시스템(RCMS) 연계 모듈 변경 사업.hwp) 범위에서는 '보유인력' 항목의 세부 배점표(예: 4대 보험 납입 기준 보유인력 20명에 대한 배점 및 배점한도)를 확인할 수 없습니다. 원문 전체의 평가항목·배점표(또는 제안서 평가 기준 표)를 확인해야 합니다.

근거: 광주과학기술원_실시간통합연구비관리시스템(RCMS) 연계 모듈 변경 사업.hwp


[visual-hwp-table-003] 국방과학연구소 기록관리시스템 사업의 요구사항 목록 전체를 기준으로, 요구사항 수가 가장 많은 구분과 가장 적은 구분(동률 포함)은 무엇이며 각각 몇 건인가? 총 요구사항 수도 함께 답하라.
가장 많은 구분: 기능 요구사항(System Function Requirement) — 21건  
가장 적은 구분(동률 포함): 전략 및 방법론 요구사항(Strategies/Methodol

In [ ]:
# 테스트 — 표/그림 등 시각자료 기반 질문

visual_test2_recheck = [
    ("visual-pdf-table-001", "서울시립대학교의 학업성취도 다차원 종단분석 통합시스템 1차 고도화 용역에서, 유사사업 최대 실적의 규모비율이 정확히 70%라면 수행실적 금액의 환산점수는 몇 점인가?"),
    ("visual-pdf-table-002", "기초과학연구원의 중이온가속기용 극저온시스템 운전 용역에서, 연구원 승인이 필요하면서 '발생한 경우에만' 제출하는 문서는 무엇이며 제출·회람·저장 조건은 무엇인가?"),
    ("visual-pdf-figure-001", "서울시 지도정보 플랫폼 시스템 개념도에서 내부 지도정보 플랫폼으로 들어오는 데이터 종류 3가지와 시민용 스마트서울맵이 제공하는 대표 지도 서비스 2가지를 각각 말해라."),
    ("visual-pdf-table-003", "서울시립대학교 학업성취도 다차원 종단분석 통합시스템 용역의 신인도 가점표에서 '약자기업 지원 및 정책적 지원' 항목 중 1점 미만인 항목만 점수별로 묶어라."),
    ("visual-pdf-figure-002", "고려대학교 차세대 포털·학사 정보시스템의 목표시스템 구성도에서 왼쪽 접근 채널 4개와 오른쪽 외부연계 대상 10개를 모두 말해라."),
]

for cid, q in visual_test2_recheck:
    answer = ask_rfp_final(q)
    print(f"[{cid}] {q}")
    print(answer)
    print("\n")

[visual-pdf-table-001] 서울시립대학교의 학업성취도 다차원 종단분석 통합시스템 1차 고도화 용역에서, 유사사업 최대 실적의 규모비율이 정확히 70%라면 수행실적 금액의 환산점수는 몇 점인가?
규모비율 70%는 60% 이상 ~ 80% 미만 구간에 해당하므로 수행실적 금액의 환산점수는 2.4점입니다.

근거: 서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf


[visual-pdf-table-002] 기초과학연구원의 중이온가속기용 극저온시스템 운전 용역에서, 연구원 승인이 필요하면서 '발생한 경우에만' 제출하는 문서는 무엇이며 제출·회람·저장 조건은 무엇인가?
문서: Lesson learn(특이사항·위험상황 발생 시 작성) — 연구원 승인 필요(O)

제출·회람·저장 조건:
- 제출시점: 발생한 경우에 한해 해당 월 기성(기성 신청) 제출 시 함께 제출  
- 제출형태: 문서 파일로 제출  
- 회람: 운전 인력 전원에게 회람 후 확인 서명 필수  
- 저장: 수치 기록을 위해 계약자가 준비한 PC에 상시 저장

근거: 기초과학연구원_2025년도 중이온가속기용 극저온시스템 운전 용역 과업지시서 (표 13)


[visual-pdf-figure-001] 서울시 지도정보 플랫폼 시스템 개념도에서 내부 지도정보 플랫폼으로 들어오는 데이터 종류 3가지와 시민용 스마트서울맵이 제공하는 대표 지도 서비스 2가지를 각각 말해라.
내부로 들어오는 데이터 종류(3가지)
- 도시생활지도(주제별/테마별 지도 콘텐츠)  
- 대기질 실시간·시계열 데이터(국가측정망·간이측정기·S-DoT 센서 연계)  
- 가로수 트리맵/3천만 그루 나무심기 트리맵 등 대용량 식생(가로수) 지도 데이터

시민용 스마트서울맵이 제공하는 대표 지도 서비스(2가지)
- 도시생활지도(시/구 부서·시민이 함께 등록·활용하는 테마별 지도)  
- 다국어지도(영어·일본어·중국어 간체 등 다국어 지도 서비스)

근거: 서울특별시_2024년 지도정보 플랫폼 및 전문활용 연계 

In [ ]:
# 코퍼스 통계 데이터 로드

import pandas as pd
import pickle

with open(DATA_DIR / 'merged_docs.pkl', 'rb') as f:
    merged_df = pickle.load(f)

print(merged_df.shape)
print(merged_df.columns.tolist())

(98, 34)
['공고 번호', '공고 차수', '사업명', '사업 금액', '발주 기관', '공개 일자', '입찰 참여 시작일', '입찰 참여 마감일', '사업 요약', '파일형식', '파일명', '텍스트', 'doc_id', 'budget_unknown', '사업_금액_정제', '사업_금액_출처', '사업_금액_후보텍스트', '공개 일자_dt', '입찰 참여 시작일_dt', '입찰 참여 마감일_dt', '입찰참여시작일_추정', '입찰참여마감일_결측', '입찰참여마감일_정제', '입찰참여마감일_출처', '입찰참여마감일_후보텍스트', '공고번호_결측', '사업명_태그', 'text', 'n_tables', 'source', 'doc_type', 'doc_type_confidence', 'doc_type_reason', 'parse_note']


In [ ]:
format_counts = merged_df['파일형식'].value_counts()
print("analytics-001")
print(format_counts)
for fmt, cnt in format_counts.items():
    print(f"{fmt}: {cnt}건 ({cnt/98*100:.2f}%)")

print()

amt = merged_df['사업 금액']
print("analytics-002")
print(f"양수: {(amt > 0).sum()}건")
print(f"결측(NaN): {amt.isna().sum()}건")
print(f"0원: {(amt == 0).sum()}건")

analytics-001
파일형식
hwp    94
pdf     4
Name: count, dtype: int64
hwp: 94건 (95.92%)
pdf: 4건 (4.08%)

analytics-002
양수: 91건
결측(NaN): 1건
0원: 6건


In [ ]:
amt2 = merged_df['사업_금액_정제']
print(f"정제 컬럼 기준 - 양수: {(amt2 > 0).sum()}건")
print(f"정제 컬럼 기준 - 결측(NaN): {amt2.isna().sum()}건")
print(f"정제 컬럼 기준 - 0원: {(amt2 == 0).sum()}건")

print(f"\n budget_unknown True 개수: {merged_df['budget_unknown'].sum()}")

정제 컬럼 기준 - 양수: 95건
정제 컬럼 기준 - 결측(NaN): 3건
정제 컬럼 기준 - 0원: 0건

budget_unknown True 개수: 3


In [ ]:
org_counts = merged_df['발주 기관'].value_counts()
print("analytics-003")
print(org_counts.head(5))

positive = merged_df[merged_df['사업_금액_정제'] > 0]['사업_금액_정제']
mean_val = positive.mean()
median_val = positive.median()
print(f"\n analytics-004")
print(f"건수: {len(positive)}건")
print(f"평균: {mean_val:,.0f}원")
print(f"중앙값: {median_val:,.0f}원")
print(f"평균/중앙값 배율: {mean_val/median_val:.2f}배")

analytics-003
발주 기관
한국철도공사 (용역)    3
한국수자원공사        3
한국생산기술연구원      2
인천광역시          2
한국연구재단         2
Name: count, dtype: int64

analytics-004
건수: 95건
평균: 741,515,031원
중앙값: 200,000,000원
평균/중앙값 배율: 3.71배


In [ ]:
import numpy as np

q1 = positive.quantile(0.25, interpolation='linear')
q2 = positive.quantile(0.50, interpolation='linear')
q3 = positive.quantile(0.75, interpolation='linear')
iqr = q3 - q1
print("analytics-005")
print(f"Q1: {q1:,.0f}원")
print(f"Q2: {q2:,.0f}원")
print(f"Q3: {q3:,.0f}원")
print(f"IQR: {iqr:,.0f}원")

sorted_desc = positive.sort_values(ascending=False)
top10 = sorted_desc.head(10)
total_sum = positive.sum()
top10_sum = top10.sum()
print(f"\n analytics-007")
print(f"상위 10건 합계: {top10_sum:,.0f}원")
print(f"전체 합계: {total_sum:,.0f}원")
print(f"비중: {top10_sum/total_sum*100:.2f}%")
print(f"경계금액(10번째 값): {top10.min():,.0f}원")

upper_fence = q3 + 1.5 * iqr
outliers = positive[positive > upper_fence]
print(f"\n analytics-008")
print(f"상단 기준: {upper_fence:,.0f}원 초과")
print(f"이상치 건수: {len(outliers)}건")
print(f"건수 비중: {len(outliers)/len(positive)*100:.2f}%")
print(f"금액 합계: {outliers.sum():,.0f}원")
print(f"금액 비중: {outliers.sum()/total_sum*100:.2f}%")

without_outliers = positive[positive <= upper_fence]
mean_wo = without_outliers.mean()
median_wo = without_outliers.median()
print(f"\n analytics-010")
print(f"제외 후 건수: {len(without_outliers)}건")
print(f"평균: {mean_wo:,.0f}원 (전체 대비 {(1-mean_wo/mean_val)*100:.2f}% 낮음)")
print(f"중앙값: {median_wo:,.0f}원 (전체 대비 {(1-median_wo/median_val)*100:.2f}% 낮음)")

analytics-005
Q1: 88,500,000원
Q2: 200,000,000원
Q3: 493,516,720원
IQR: 405,016,720원

 analytics-007
상위 10건 합계: 47,166,642,413원
전체 합계: 70,443,927,955원
비중: 66.96%
경계금액(10번째 값): 1,095,991,600원

 analytics-008
상단 기준: 1,101,041,800원 초과
이상치 건수: 9건
건수 비중: 9.47%
금액 합계: 46,070,650,813원
금액 비중: 65.40%

 analytics-010
제외 후 건수: 86건
평균: 283,410,199원 (전체 대비 61.78% 낮음)
중앙값: 188,471,500원 (전체 대비 5.76% 낮음)


In [ ]:
import json

with open(DATA_DIR / 'project-category-v1.jsonl', encoding='utf-8') as f:
    cat_items = [json.loads(l) for l in f if l.strip()]

cat_df = pd.DataFrame(cat_items)
print(cat_df.shape)
print(cat_df[['project_name', 'project_category_v1', 'amount_won']].head())

(98, 11)
                               project_name      project_category_v1  \
0  한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화  enhancement_improvement   
1        2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선  enhancement_improvement   
2                EIP3.0 고압가스 안전관리 시스템 구축 용역        new_build_rebuild   
3                      도시계획위원회 통합관리시스템 구축용역        new_build_rebuild   
4              봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)  enhancement_improvement   

    amount_won  
0  130000000.0  
1  129300000.0  
2   40000000.0  
3  150000000.0  
4  900000000.0  


In [ ]:
merged_df['사업명_정규화'] = merged_df['사업명'].astype(str).str.strip()
cat_df['project_name_정규화'] = cat_df['project_name'].astype(str).str.strip()

merged_with_cat = merged_df.merge(cat_df[['project_name_정규화', 'project_category_v1']],
                                    left_on='사업명_정규화', right_on='project_name_정규화', how='left')
print(f"매칭 성공: {merged_with_cat['project_category_v1'].notna().sum()}건 / 전체 {len(merged_with_cat)}건")

매칭 성공: 98건 / 전체 98건


In [ ]:
for cat, label in [('new_build_rebuild', '신규 구축·재구축'), ('enhancement_improvement', '고도화·개선')]:
    subset = merged_with_cat[merged_with_cat['project_category_v1'] == cat]
    positive_subset = subset[subset['사업_금액_정제'] > 0]['사업_금액_정제']
    print(f"{label}: {len(subset)}건 (금액 양수 {len(positive_subset)}건), 중앙값: {positive_subset.median():,.0f}원")

print()

print("analytics-009")
for cat in merged_with_cat['project_category_v1'].dropna().unique():
    subset = merged_with_cat[merged_with_cat['project_category_v1'] == cat]
    positive_subset = subset[subset['사업_금액_정제'] > 0]['사업_금액_정제']
    if len(positive_subset) >= 5:
        mean_c = positive_subset.mean()
        median_c = positive_subset.median()
        ratio = mean_c / median_c
        print(f"{cat}: 건수={len(positive_subset)}, 평균={mean_c:,.0f}원, 중앙값={median_c:,.0f}원, 비율={ratio:.2f}")

신규 구축·재구축: 44건 (금액 양수 42건), 중앙값: 200,000,000원
고도화·개선: 30건 (금액 양수 29건), 중앙값: 140,000,000원

analytics-009
enhancement_improvement: 건수=29, 평균=277,353,576원, 중앙값=140,000,000원, 비율=1.98
new_build_rebuild: 건수=42, 평균=1,072,946,390원, 중앙값=200,000,000원, 비율=5.36
operations_maintenance: 건수=12, 평균=332,387,201원, 중앙값=216,150,000원, 비율=1.54
planning_consulting: 건수=12, 평균=1,112,356,622원, 중앙값=385,125,984원, 비율=2.89


In [ ]:
def analytics_summary(merged_with_cat):
    positive = merged_with_cat[merged_with_cat['사업_금액_정제'] > 0]['사업_금액_정제']

    result = {}

    result['파일형식_분포'] = merged_with_cat['파일형식'].value_counts().to_dict()

    amt = merged_with_cat['사업_금액_정제']
    result['금액_상태'] = {
        '양수': int((amt > 0).sum()),
        '결측': int(amt.isna().sum()),
        '0원': int((amt == 0).sum())
    }

    org_counts = merged_with_cat['발주 기관'].value_counts()
    max_count = org_counts.max()
    result['최다_발주기관'] = org_counts[org_counts == max_count].to_dict()

    mean_val = positive.mean()
    median_val = positive.median()
    result['평균_중앙값'] = {
        '평균': round(mean_val),
        '중앙값': round(median_val),
        '배율': round(mean_val / median_val, 2)
    }

    q1 = positive.quantile(0.25)
    q3 = positive.quantile(0.75)
    iqr = q3 - q1
    result['사분위수'] = {
        'Q1': round(q1), 'Q2': round(median_val), 'Q3': round(q3), 'IQR': round(iqr)
    }

    if 'project_category_v1' in merged_with_cat.columns:
        cat_stats = {}
        for cat in merged_with_cat['project_category_v1'].dropna().unique():
            subset = merged_with_cat[merged_with_cat['project_category_v1'] == cat]
            pos_subset = subset[subset['사업_금액_정제'] > 0]['사업_금액_정제']
            cat_stats[cat] = {
                '전체건수': len(subset),
                '금액양수건수': len(pos_subset),
                '평균': round(pos_subset.mean()) if len(pos_subset) > 0 else None,
                '중앙값': round(pos_subset.median()) if len(pos_subset) > 0 else None,
                '평균중앙값비율': round(pos_subset.mean() / pos_subset.median(), 2) if len(pos_subset) > 0 else None
            }
        result['카테고리별_통계'] = cat_stats

    sorted_desc = positive.sort_values(ascending=False)
    top10 = sorted_desc.head(10)
    total_sum = positive.sum()
    result['상위10건'] = {
        '합계': round(top10.sum()),
        '비중': round(top10.sum() / total_sum * 100, 2),
        '경계금액': round(top10.min())
    }

    upper_fence = q3 + 1.5 * iqr
    outliers = positive[positive > upper_fence]
    without_outliers = positive[positive <= upper_fence]
    result['이상치'] = {
        '상단기준': round(upper_fence),
        '건수': len(outliers),
        '건수비중': round(len(outliers) / len(positive) * 100, 2),
        '금액합계': round(outliers.sum()),
        '금액비중': round(outliers.sum() / total_sum * 100, 2),
        '제외후_평균': round(without_outliers.mean()),
        '제외후_중앙값': round(without_outliers.median()),
        '평균_감소율': round((1 - without_outliers.mean()/mean_val) * 100, 2),
        '중앙값_감소율': round((1 - without_outliers.median()/median_val) * 100, 2)
    }

    return result

In [ ]:
summary = analytics_summary(merged_with_cat)

print(json.dumps(summary, ensure_ascii=False, indent=2))

{
  "파일형식_분포": {
    "hwp": 94,
    "pdf": 4
  },
  "금액_상태": {
    "양수": 95,
    "결측": 3,
    "0원": 0
  },
  "최다_발주기관": {
    "한국철도공사 (용역)": 3,
    "한국수자원공사": 3
  },
  "평균_중앙값": {
    "평균": 741515031,
    "중앙값": 200000000,
    "배율": 3.71
  },
  "사분위수": {
    "Q1": 88500000,
    "Q2": 200000000,
    "Q3": 493516720,
    "IQR": 405016720
  },
  "카테고리별_통계": {
    "enhancement_improvement": {
      "전체건수": 30,
      "금액양수건수": 29,
      "평균": 277353576,
      "중앙값": 140000000,
      "평균중앙값비율": 1.98
    },
    "new_build_rebuild": {
      "전체건수": 44,
      "금액양수건수": 42,
      "평균": 1072946390,
      "중앙값": 200000000,
      "평균중앙값비율": 5.36
    },
    "operations_maintenance": {
      "전체건수": 12,
      "금액양수건수": 12,
      "평균": 332387201,
      "중앙값": 216150000,
      "평균중앙값비율": 1.54
    },
    "planning_consulting": {
      "전체건수": 12,
      "금액양수건수": 12,
      "평균": 1112356622,
      "중앙값": 385125984,
      "평균중앙값비율": 2.89
    }
  },
  "상위10건": {
    "합계": 47166642413,
    "비중": 66.96,
    "

In [ ]:
# 코퍼스 통계형 질문문

with open(DATA_DIR / 'corpus-analytics-qa.jsonl', encoding='utf-8') as f:
    analytics_questions = [json.loads(l) for l in f if l.strip()]

summary = analytics_summary(merged_with_cat)

def answer_analytics(case_id, summary):
    if case_id == 'analytics-001':
        return f"HWP {summary['파일형식_분포']['hwp']}건({summary['파일형식_분포']['hwp']/98*100:.2f}%), PDF {summary['파일형식_분포']['pdf']}건({summary['파일형식_분포']['pdf']/98*100:.2f}%)"
    elif case_id == 'analytics-002':
        s = summary['금액_상태']
        return f"양수 {s['양수']}건, 결측 {s['결측']}건, 0원 {s['0원']}건. 금액 통계에는 양수 건만 사용."
    elif case_id == 'analytics-003':
        return f"공동 1위: {', '.join(f'{k}({v}건)' for k, v in summary['최다_발주기관'].items())}"
    elif case_id == 'analytics-004':
        s = summary['평균_중앙값']
        return f"평균 {s['평균']:,}원, 중앙값 {s['중앙값']:,}원, 평균은 중앙값의 {s['배율']}배"
    elif case_id == 'analytics-005':
        s = summary['사분위수']
        return f"Q1 {s['Q1']:,}원, Q2 {s['Q2']:,}원, Q3 {s['Q3']:,}원, IQR {s['IQR']:,}원"
    elif case_id == 'analytics-006':
        c = summary['카테고리별_통계']
        return f"신규구축·재구축 {c['new_build_rebuild']['전체건수']}건(중앙값 {c['new_build_rebuild']['중앙값']:,}원), 고도화·개선 {c['enhancement_improvement']['전체건수']}건(중앙값 {c['enhancement_improvement']['중앙값']:,}원)"
    elif case_id == 'analytics-007':
        s = summary['상위10건']
        return f"상위 10건 비중 {s['비중']}%, 경계금액 {s['경계금액']:,}원"
    elif case_id == 'analytics-008':
        s = summary['이상치']
        return f"이상치 {s['건수']}건, 건수비중 {s['건수비중']}%, 금액비중 {s['금액비중']}%"
    elif case_id == 'analytics-009':
        c = summary['카테고리별_통계']
        best = max(c.items(), key=lambda x: x[1]['평균중앙값비율'] if x[1]['금액양수건수'] >= 5 else -1)
        return f"{best[0]}: 비율 {best[1]['평균중앙값비율']}, 평균 {best[1]['평균']:,}원, 중앙값 {best[1]['중앙값']:,}원"
    elif case_id == 'analytics-010':
        s = summary['이상치']
        return f"이상치 제외 평균 {s['제외후_평균']:,}원({s['평균_감소율']}% 낮음), 중앙값 {s['제외후_중앙값']:,}원({s['중앙값_감소율']}% 낮음)"
    return "미지원 질문"

for q in analytics_questions:
    print(f"[{q['case_id']}] {q['question']}")
    print(f"  답: {answer_analytics(q['case_id'], summary)}")
    print()

[analytics-001] refined 98개 문서는 HWP와 PDF가 각각 몇 건이며, 각 형식의 비율은 얼마인가?
  답: HWP 94건(95.92%), PDF 4건(4.08%)

[analytics-002] refined 98개 문서의 사업 금액은 양수·결측·0원이 각각 몇 건인가? 금액 통계에는 어떤 행을 사용해야 하는가?
  답: 양수 95건, 결측 3건, 0원 0건. 금액 통계에는 양수 건만 사용.

[analytics-003] refined 말뭉치에서 문서 수가 가장 많은 발주기관은 어디이며 각각 몇 건인가? 공동 1위가 있으면 모두 답하라.
  답: 공동 1위: 한국철도공사 (용역)(3건), 한국수자원공사(3건)

[analytics-004] 사업 금액이 양수인 95개 사업의 평균과 중앙값은 얼마이며, 평균은 중앙값의 몇 배인가?
  답: 평균 741,515,031원, 중앙값 200,000,000원, 평균은 중앙값의 3.71배

[analytics-005] 사업 금액 양수 95건의 1사분위수(Q1), 중앙값(Q2), 3사분위수(Q3), 사분위범위(IQR)는 각각 얼마인가?
  답: Q1 88,500,000원, Q2 200,000,000원, Q3 493,516,720원, IQR 405,016,720원

[analytics-006] project_category_v1 기준으로 '신규 구축·재구축'과 '고도화·개선'은 각각 몇 건이며, 사업 금액 양수 건의 중앙값은 얼마인가?
  답: 신규구축·재구축 44건(중앙값 200,000,000원), 고도화·개선 30건(중앙값 140,000,000원)

[analytics-007] 사업 금액 양수 95건에서 상위 10%를 올림 처리해 10건으로 잡으면, 이 10건이 전체 금액에서 차지하는 비중과 경계 금액은 얼마인가?
  답: 상위 10건 비중 66.96%, 경계금액 1,095,991,600원

[analytics-008] Tukey 상단 이상치 기준(Q3 + 1.5×IQR)을 적용하면 사업 금액 이상치는 